# 데이터 특성 파악 및 집계 (Aggregation)

베이스라인 이후 피처 엔지니어링을 아래 4단계로 나눠 진행합니다.

1. **집계 (Aggregation)** — 이 노트북. 격자/터빈/시간 단위로 흩어진 원천 데이터를 예측 시각(`forecast_kst_dtm` 또는 `kst_dtm`) 단위로 모읍니다.
2. **일반화 (Generalization)** — 다음 노트북. 그룹/터빈별로 흩어진 집계 결과를 공통 스키마로 정리하고 이상치를 처리합니다.
3. **정규화 (Normalization)** — 스케일이 다른 변수들을 학습에 적합한 범위로 변환합니다.
4. **평활화 (Smoothing)** — 시계열 노이즈를 줄이기 위한 이동평균/필터를 적용합니다.

이 노트북은 1단계(집계)만 다루며, 결과는 `agg/` 폴더에 중간 산출물(CSV)로 저장해 다음 단계 노트북에서 이어받아 사용합니다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(".")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
INFO_DIR = DATA_DIR / "INFO"
AGG_DIR = DATA_DIR / "agg"
AGG_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 20)

## 1. 원천 데이터 개요

각 파일을 로드하고 행/열 수, 시간 범위를 한눈에 정리합니다.

In [3]:
train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(INFO_DIR / "sample_submission.csv", encoding="utf-8-sig")
ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")
scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")

raw_files = {
    "train_labels": (train_labels, "kst_dtm"),
    "sample_submission": (sample_submission, "forecast_kst_dtm"),
    "ldaps_train": (ldaps_train, "forecast_kst_dtm"),
    "gfs_train": (gfs_train, "forecast_kst_dtm"),
    "ldaps_test": (ldaps_test, "forecast_kst_dtm"),
    "gfs_test": (gfs_test, "forecast_kst_dtm"),
    "scada_vestas_train": (scada_vestas, "kst_dtm"),
    "scada_unison_train": (scada_unison, "kst_dtm"),
}

summary_rows = []
for name, (df, dt_col) in raw_files.items():
    dt = pd.to_datetime(df[dt_col])
    summary_rows.append({
        "file": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "start": dt.min(),
        "end": dt.max(),
    })

summary = pd.DataFrame(summary_rows)
summary

,file,rows,cols,start,end
0,train_labels,26304,4,2022-01-01 01:00:00,2025-01-01
1,sample_submission,8760,5,2025-01-01 01:00:00,2026-01-01
2,ldaps_train,420864,35,2022-01-01 01:00:00,2025-01-01
3,gfs_train,236736,40,2022-01-01 01:00:00,2025-01-01
4,ldaps_test,140160,35,2025-01-01 01:00:00,2026-01-01
5,gfs_test,78840,40,2025-01-01 01:00:00,2026-01-01
6,scada_vestas_train,157819,37,2022-01-01 01:00:00,2025-01-01
7,scada_unison_train,105264,16,2023-01-01 00:10:00,2025-01-01


In [4]:
for name, (df, _) in raw_files.items():
    print(f"=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===")
    print("컬럼명:", list(df.columns))
    stats = pd.concat([df.dtypes.rename("dtype"), df.describe(include="all").T], axis=1)
    display(stats)
    print()


=== train_labels: 26,304 rows x 4 cols ===
컬럼명: ['kst_dtm', 'kpx_group_1', 'kpx_group_2', 'kpx_group_3']


,dtype,count,unique,top,freq,mean,std,min,25%,50%,75%,max
kst_dtm,str,26304,26304,2022-01-01 01:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kpx_group_1,float64,26200.0,NaN,NaN,NaN,6621.981125,6582.443128,0.0,549.6,4252.168,12206.905,21275.305
kpx_group_2,float64,26201.0,NaN,NaN,NaN,7076.842859,7001.146068,0.0,549.6,4382.337,13508.589,21362.084
kpx_group_3,float64,17538.0,NaN,NaN,NaN,5563.819649,6294.582901,0.0,159.095,2719.074,9979.579,21130.674



=== sample_submission: 8,760 rows x 5 cols ===
컬럼명: ['forecast_id', 'forecast_kst_dtm', 'kpx_group_1', 'kpx_group_2', 'kpx_group_3']


,dtype,count,unique,top,freq,mean,std,min,25%,50%,75%,max
forecast_id,str,8760,8760,forecast_0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
forecast_kst_dtm,str,8760,8760,2025-01-01 01:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kpx_group_1,int64,8760.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kpx_group_2,int64,8760.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kpx_group_3,int64,8760.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== ldaps_train: 420,864 rows x 35 cols ===
컬럼명: ['forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id', 'latitude', 'longitude', 'heightAboveGround_10_10u', 'heightAboveGround_10_10v', 'heightAboveGround_50_50MUmax', 'heightAboveGround_50_50MUmin', 'heightAboveGround_50_50MVmax', 'heightAboveGround_50_50MVmin', 'heightAboveGround_5_XBLWS', 'heightAboveGround_5_YBLWS', 'heightAboveGround_2_t', 'heightAboveGround_2_dpt', 'heightAboveGround_2_r', 'heightAboveGround_2_q', 'surface_0_sp', 'meanSea_0_prmsl', 'etc_0_blh', 'surface_0_NDNSW', 'surface_0_NDNLW', 'heightAboveGround_2_SWDIR', 'heightAboveGround_2_SWDIF', 'etc_0_hcc', 'etc_0_mcc', 'etc_0_lcc', 'etc_0_VLCDC', 'surface_0_avg_lsprate', 'surface_0_lssrate', 'surface_0_ncpcp', 'surface_0_snol', 'surface_0_SNOM', 'surface_0_lsm', 'surface_0_h']


...

,dtype,count,unique,top,freq,mean,std,min,25%,50%,75%,max
forecast_kst_dtm,str,78840,8760,2025-01-01 01:00:00,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
data_available_kst_dtm,str,78840,365,2024-12-31 13:00:00,216,NaN,NaN,NaN,NaN,NaN,NaN,NaN
grid_id,int64,78840.0,NaN,NaN,NaN,5.0,2.582005,1.0,3.0,5.0,7.0,9.0
latitude,float64,78840.0,NaN,NaN,NaN,37.25,0.204125,37.0,37.0,37.25,37.5,37.5
longitude,float64,78840.0,NaN,NaN,NaN,129.0,0.204125,128.75,128.75,129.0,129.25,129.25
heightAboveGround_10_10u,float64,78840.0,NaN,NaN,NaN,1.529077,2.386495,-11.45808,-0.041829,1.518143,2.832349,20.35683
heightAboveGround_10_10v,float64,78840.0,NaN,NaN,NaN,0.277444,1.626097,-14.282737,-0.599963,0.41764,1.134816,12.810869
heightAboveGround_80_u,float64,78840.0,NaN,NaN,NaN,2.399277,3.585652,-13.479356,0.055854,2.145956,4.106033,26.485977
heightAboveGround_80_v,float64,78840.0,NaN,NaN,NaN,0.343528,2.228637,-17.039068,-0.842317,0.47272,1.498981,16.00849
heightAboveGround_100_100u,float64,78840.0,NaN,NaN,NaN,2.523777,3.762073,-13.797676,0.053231,2.224679,4.305187,27.09684



=== scada_vestas_train: 157,819 rows x 37 cols ===
컬럼명: ['kst_dtm', 'vestas_wtg01_power_kw10m', 'vestas_wtg02_power_kw10m', 'vestas_wtg03_power_kw10m', 'vestas_wtg04_power_kw10m', 'vestas_wtg05_power_kw10m', 'vestas_wtg06_power_kw10m', 'vestas_wtg07_power_kw10m', 'vestas_wtg08_power_kw10m', 'vestas_wtg09_power_kw10m', 'vestas_wtg10_power_kw10m', 'vestas_wtg11_power_kw10m', 'vestas_wtg12_power_kw10m', 'vestas_wtg01_ws', 'vestas_wtg02_ws', 'vestas_wtg03_ws', 'vestas_wtg04_ws', 'vestas_wtg05_ws', 'vestas_wtg06_ws', 'vestas_wtg07_ws', 'vestas_wtg08_ws', 'vestas_wtg09_ws', 'vestas_wtg10_ws', 'vestas_wtg11_ws', 'vestas_wtg12_ws', 'vestas_wtg01_wd', 'vestas_wtg02_wd', 'vestas_wtg03_wd', 'vestas_wtg04_wd', 'vestas_wtg05_wd', 'vestas_wtg06_wd', 'vestas_wtg07_wd', 'vestas_wtg08_wd', 'vestas_wtg09_wd', 'vestas_wtg10_wd', 'vestas_wtg11_wd', 'vestas_wtg12_wd']


,dtype,count,unique,top,freq,mean,std,min,25%,50%,75%,max
kst_dtm,str,157819,157819,2022-01-01 01:00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vestas_wtg01_power_kw10m,int64,157819.0,NaN,NaN,NaN,195.472066,712805.870174,-42512957.0,5.0,102.0,371.0,42512957.0
vestas_wtg02_power_kw10m,int64,157819.0,NaN,NaN,NaN,201.289148,691914.947412,-41959813.0,9.0,107.0,389.0,41959813.0
vestas_wtg03_power_kw10m,int64,157819.0,NaN,NaN,NaN,217.457809,786801.148854,-46559445.0,12.0,126.0,431.0,46559445.0
vestas_wtg04_power_kw10m,int64,157819.0,NaN,NaN,NaN,186.633777,480318.380515,-40642699.0,2.0,97.0,345.0,40642699.0
vestas_wtg05_power_kw10m,int64,157819.0,NaN,NaN,NaN,153.188786,456684.663624,-33717814.0,0.0,58.0,268.0,33717814.0
vestas_wtg06_power_kw10m,int64,157819.0,NaN,NaN,NaN,158.188127,478422.426755,-34184409.0,3.0,72.0,274.0,34184409.0
vestas_wtg07_power_kw10m,int64,157819.0,NaN,NaN,NaN,138.37761,418071.325751,-29770834.0,0.0,52.0,233.0,29770834.0
vestas_wtg08_power_kw10m,int64,157819.0,NaN,NaN,NaN,185.373447,802538.179135,-42301001.0,5.0,95.0,350.0,42300996.0
vestas_wtg09_power_kw10m,int64,157819.0,NaN,NaN,NaN,199.256205,580428.740812,-43556999.0,5.0,102.0,385.0,43556994.0



=== scada_unison_train: 105,264 rows x 16 cols ===
컬럼명: ['kst_dtm', 'unison_wtg01_power_kw10m', 'unison_wtg02_power_kw10m', 'unison_wtg03_power_kw10m', 'unison_wtg04_power_kw10m', 'unison_wtg05_power_kw10m', 'unison_wtg01_ws', 'unison_wtg02_ws', 'unison_wtg03_ws', 'unison_wtg04_ws', 'unison_wtg05_ws', 'unison_wtg01_wd', 'unison_wtg02_wd', 'unison_wtg03_wd', 'unison_wtg04_wd', 'unison_wtg05_wd']


,dtype,count,unique,top,freq,mean,std,min,25%,50%,75%,max
kst_dtm,str,105264,105264,2023-01-01 00:10:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unison_wtg01_power_kw10m,float64,105056.0,NaN,NaN,NaN,162.753741,213.965741,0.0,0.0,67.0,289.0,800.0
unison_wtg02_power_kw10m,float64,103902.0,NaN,NaN,NaN,145.411022,209.503968,0.0,0.0,34.5,200.0,1000.0
unison_wtg03_power_kw10m,float64,104912.0,NaN,NaN,NaN,201.070268,241.747608,0.0,0.0,100.0,395.0,900.0
unison_wtg04_power_kw10m,float64,105114.0,NaN,NaN,NaN,196.390233,239.218988,0.0,0.0,100.0,362.75,800.0
unison_wtg05_power_kw10m,float64,105122.0,NaN,NaN,NaN,221.543216,262.719079,0.0,0.0,100.0,416.0,800.0
unison_wtg01_ws,float64,103798.0,NaN,NaN,NaN,5.509658,3.208078,0.0,2.81,4.89,7.62,22.16
unison_wtg02_ws,float64,103840.0,NaN,NaN,NaN,5.183158,3.599933,0.15,2.5,4.1,6.87,26.52
unison_wtg03_ws,float64,104817.0,NaN,NaN,NaN,5.996033,3.896123,0.11,2.73,5.07,8.45,25.04
unison_wtg04_ws,float64,104884.0,NaN,NaN,NaN,6.049325,3.927545,0.0,2.77,5.19,8.45,27.98


## 2. 결측치 확인

기상 예보 데이터(LDAPS/GFS)는 격자 관측값이라 결측이 거의 없을 것으로 예상되지만, SCADA는 실측 센서 데이터라
결측/오류가 섞여 있을 가능성이 높습니다. 파일별 결측 비율을 확인합니다.

In [3]:
missing_summary = {}
for name, (df, _) in raw_files.items():
    na_ratio = df.isna().mean()
    missing_summary[name] = na_ratio[na_ratio > 0]

for name, na in missing_summary.items():
    if len(na) == 0:
        print(f"[{name}] 결측 없음")
    else:
        print(f"[{name}] 결측 비율(상위 5개)")
        print(na.sort_values(ascending=False).head(5))
    print()

[train_labels] 결측 비율(상위 5개)
kpx_group_3    0.333257
kpx_group_1    0.003954
kpx_group_2    0.003916
dtype: float64

[sample_submission] 결측 없음

[ldaps_train] 결측 없음

[gfs_train] 결측 없음

[ldaps_test] 결측 비율(상위 5개)
heightAboveGround_50_50MUmax    0.000342
heightAboveGround_50_50MUmin    0.000342
heightAboveGround_50_50MVmax    0.000342
heightAboveGround_50_50MVmin    0.000342
meanSea_0_prmsl                 0.000342
dtype: float64

[gfs_test] 결측 없음

[scada_vestas_train] 결측 없음

[scada_unison_train] 결측 비율(상위 5개)
unison_wtg01_ws             0.013927
unison_wtg02_ws             0.013528
unison_wtg05_ws             0.013100
unison_wtg02_power_kw10m    0.012939
unison_wtg02_wd             0.012863
dtype: float64



## 3. 이상치 확인 (SCADA 발전량)

VESTAS 터빈은 설비용량이 3.6MW(=3,600kW)인데, `power_kw10m` 값 중 이 범위를 크게 벗어나는 수백만 단위의
극단값이 존재합니다. 센서/통신 오류로 보이는 값으로, 그대로 평균에 반영하면 집계값이 심하게 왜곡됩니다.

이 노트북에서는 **이상치를 수정하지 않고 존재 여부만 확인**합니다. 실제 보정은 다음 단계인 일반화(Generalization)
노트북에서 처리합니다.

In [4]:
vestas_power_cols = [c for c in scada_vestas.columns if c.endswith("power_kw10m")]
capacity_per_turbine_kw = 3600  # VESTAS 1기 설비용량 3.6MW

outlier_counts = (scada_vestas[vestas_power_cols].abs() > capacity_per_turbine_kw).sum()
print("설비용량(3,600kW)을 초과하는 값 개수 (터빈별):")
print(outlier_counts)

print()
print("wtg01 극단값 예시:")
mask = scada_vestas["vestas_wtg01_power_kw10m"].abs() > capacity_per_turbine_kw
print(scada_vestas.loc[mask, ["kst_dtm", "vestas_wtg01_power_kw10m"]].head())

print()
print("풍향(wd) 표기 범위 비교 - VESTAS: 0~360, UNISON: -180~180 (부호 있는 표기)")
print("vestas_wtg01_wd range:", scada_vestas["vestas_wtg01_wd"].min(), "~", scada_vestas["vestas_wtg01_wd"].max())
print("unison_wtg01_wd range:", scada_unison["unison_wtg01_wd"].min(), "~", scada_unison["unison_wtg01_wd"].max())

설비용량(3,600kW)을 초과하는 값 개수 (터빈별):
vestas_wtg01_power_kw10m     74
vestas_wtg02_power_kw10m     70
vestas_wtg03_power_kw10m     88
vestas_wtg04_power_kw10m     44
vestas_wtg05_power_kw10m     72
vestas_wtg06_power_kw10m     60
vestas_wtg07_power_kw10m     68
vestas_wtg08_power_kw10m    102
vestas_wtg09_power_kw10m     52
vestas_wtg10_power_kw10m     68
vestas_wtg11_power_kw10m     98
vestas_wtg12_power_kw10m     72
dtype: int64

wtg01 극단값 예시:
                  kst_dtm  vestas_wtg01_power_kw10m
7432  2022-02-21 15:40:00                 -15326974
7433  2022-02-21 15:50:00                  15326971
9006  2022-03-04 14:00:00                 -16006532
9015  2022-03-04 15:30:00                  16006654
9016  2022-03-04 15:40:00                 -16006654

풍향(wd) 표기 범위 비교 - VESTAS: 0~360, UNISON: -180~180 (부호 있는 표기)
vestas_wtg01_wd range: 0.0 ~ 359.0
unison_wtg01_wd range: -179.99800436642 ~ 179.99237027425


## 4. 기상 데이터 격자 구조 확인

LDAPS는 16개 격자, GFS는 9개 격자가 각 예측 시각(`forecast_kst_dtm`)마다 존재합니다. 격자 수가 명세와
일치하는지, 시각별 행 수가 일정한지 확인합니다.

In [5]:
for name, df in [("ldaps_train", ldaps_train), ("gfs_train", gfs_train)]:
    grid_count = df["grid_id"].nunique()
    rows_per_dtm = df.groupby("forecast_kst_dtm").size()
    print(f"[{name}] grid_id 개수: {grid_count}, forecast_kst_dtm당 행수 unique값: {rows_per_dtm.unique()}")

[ldaps_train] grid_id 개수: 16, forecast_kst_dtm당 행수 unique값: [16]
[gfs_train] grid_id 개수: 9, forecast_kst_dtm당 행수 unique값: [9]


## 5. `info.xlsx` 기반 터빈 - KPX 그룹 매핑

`info.xlsx`는 그룹 정보가 병합 셀로 저장되어 있어, 첫 번째 터빈 행에만 `KPX그룹` 값이 채워져 있습니다.
`ffill`로 아래 행까지 그룹 값을 채운 뒤 터빈 컬럼명과 KPX 그룹을 매핑하는 딕셔너리를 만듭니다.

SCADA 집계 단계에서 이 매핑을 이용해 터빈별 값을 KPX 그룹 단위로 합산합니다.

In [6]:
info = pd.read_excel(INFO_DIR / "info.xlsx", header=2)
info = info.iloc[:, 1:]
info.columns = info.iloc[0]
info = info.iloc[1:].reset_index(drop=True)
info["KPX그룹"] = info["KPX그룹"].ffill().astype(int)

vestas_group = {}
unison_group = {}
for _, row in info.iterrows():
    maker = row["제작사"]
    turbine_no = int(row["호기"])
    group = row["KPX그룹"]
    if maker == "VESTAS":
        vestas_group[f"vestas_wtg{turbine_no:02d}"] = f"kpx_group_{group}"
    else:
        unison_group[f"unison_wtg{turbine_no:02d}"] = f"kpx_group_{group}"

print("VESTAS 터빈 -> KPX 그룹:", vestas_group)
print("UNISON 터빈 -> KPX 그룹:", unison_group)

VESTAS 터빈 -> KPX 그룹: {'vestas_wtg01': 'kpx_group_1', 'vestas_wtg02': 'kpx_group_1', 'vestas_wtg03': 'kpx_group_1', 'vestas_wtg04': 'kpx_group_1', 'vestas_wtg05': 'kpx_group_1', 'vestas_wtg06': 'kpx_group_1', 'vestas_wtg07': 'kpx_group_2', 'vestas_wtg08': 'kpx_group_2', 'vestas_wtg09': 'kpx_group_2', 'vestas_wtg10': 'kpx_group_2', 'vestas_wtg11': 'kpx_group_2', 'vestas_wtg12': 'kpx_group_2'}
UNISON 터빈 -> KPX 그룹: {'unison_wtg01': 'kpx_group_3', 'unison_wtg02': 'kpx_group_3', 'unison_wtg03': 'kpx_group_3', 'unison_wtg04': 'kpx_group_3', 'unison_wtg05': 'kpx_group_3'}


## 6. 집계 - 기상 데이터 (격자 간 통계량)

LDAPS/GFS는 `forecast_kst_dtm` 하나당 여러 격자(grid_id)가 존재하므로, 시각별로 격자 간
**평균/표준편차/최솟값/최댓값**을 집계합니다. 표준편차와 최소/최댓값은 격자 간 편차(공간적 변동성)를
나타내는 피처로, 평균만 쓰는 베이스라인보다 더 많은 정보를 담습니다.

In [7]:
def aggregate_weather_stats(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    g = df.groupby("forecast_kst_dtm")[value_cols]
    agg = pd.concat({"mean": g.mean(), "std": g.std(), "min": g.min(), "max": g.max()}, axis=1)
    agg.columns = [f"{prefix}_{col}_{stat}" for stat, col in agg.columns]
    return agg.reset_index()


ldaps_train_agg = aggregate_weather_stats(ldaps_train, "ldaps")
gfs_train_agg = aggregate_weather_stats(gfs_train, "gfs")
ldaps_test_agg = aggregate_weather_stats(ldaps_test, "ldaps")
gfs_test_agg = aggregate_weather_stats(gfs_test, "gfs")

print("ldaps_train_agg:", ldaps_train_agg.shape)
print("gfs_train_agg:", gfs_train_agg.shape)
print("ldaps_test_agg:", ldaps_test_agg.shape)
print("gfs_test_agg:", gfs_test_agg.shape)

ldaps_train_agg: (26304, 121)
gfs_train_agg: (26304, 141)
ldaps_test_agg: (8760, 121)
gfs_test_agg: (8760, 141)


## 7. 집계 - SCADA 데이터 (10분 -> 1시간, 터빈 -> KPX 그룹)

`train_labels.kst_dtm`은 **집계 구간의 종료 시각**입니다. 즉 `01:00:00`은 `00:00:01~01:00:00` 구간을 뜻합니다.
SCADA는 10분 간격(`00:10, 00:20, ..., 01:00`)으로 기록되므로, 같은 규칙에 맞춰 `dt.ceil("1h")`로 그룹 키를
잡아야 라벨과 정확히 대응됩니다 (`dt.floor`를 쓰면 한 시간 어긋납니다).

풍향(`wd`)은 순환 변수라 단순 평균이 불가능합니다 (0도와 359도의 평균이 180도가 되는 오류 방지). 이 노트북에서는
sin/cos 성분의 평균 후 `atan2`로 되돌리는 **원형 평균(circular mean)** 을 사용합니다. VESTAS(0~360)와
UNISON(-180~180)처럼 표기 범위가 달라도 sin/cos 변환에서는 동일하게 처리됩니다.

터빈별 평균 발전량은 매핑 딕셔너리를 이용해 KPX 그룹별로 합산합니다.

In [8]:
def aggregate_scada_hourly(df, turbine_prefixes, group_map):
    df = df.copy()
    df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    df["hour_end"] = df["kst_dtm"].dt.ceil("1h")

    out = pd.DataFrame({"kst_dtm": sorted(df["hour_end"].unique())})

    for prefix in turbine_prefixes:
        power_col = f"{prefix}_power_kw10m"
        ws_col = f"{prefix}_ws"
        wd_col = f"{prefix}_wd"

        g = df.groupby("hour_end")
        power_mean = g[power_col].mean()
        ws_mean = g[ws_col].mean()

        wd_rad = np.deg2rad(df[wd_col])
        sin_mean = np.sin(wd_rad).groupby(df["hour_end"]).mean()
        cos_mean = np.cos(wd_rad).groupby(df["hour_end"]).mean()
        wd_mean = (np.degrees(np.arctan2(sin_mean, cos_mean)) + 360) % 360

        out = out.merge(power_mean.rename(f"{prefix}_power_mean"), left_on="kst_dtm", right_index=True, how="left")
        out = out.merge(ws_mean.rename(f"{prefix}_ws_mean"), left_on="kst_dtm", right_index=True, how="left")
        out = out.merge(wd_mean.rename(f"{prefix}_wd_mean"), left_on="kst_dtm", right_index=True, how="left")

    for group in sorted(set(group_map.values())):
        members = [p for p, g in group_map.items() if g == group]
        power_cols = [f"{p}_power_mean" for p in members]
        out[f"{group}_power_sum"] = out[power_cols].sum(axis=1)

    return out


vestas_prefixes = [f"vestas_wtg{i:02d}" for i in range(1, 13)]
unison_prefixes = [f"unison_wtg{i:02d}" for i in range(1, 6)]

scada_vestas_agg = aggregate_scada_hourly(scada_vestas, vestas_prefixes, vestas_group)
scada_unison_agg = aggregate_scada_hourly(scada_unison, unison_prefixes, unison_group)

print("scada_vestas_agg:", scada_vestas_agg.shape)
print("scada_unison_agg:", scada_unison_agg.shape)
scada_vestas_agg[["kst_dtm", "kpx_group_1_power_sum", "kpx_group_2_power_sum"]].head()

scada_vestas_agg: (26304, 39)
scada_unison_agg: (17544, 17)


,kst_dtm,kpx_group_1_power_sum,kpx_group_2_power_sum
0,2022-01-01 01:00:00,2005.000000,1536.000000
1,2022-01-01 02:00:00,2181.666667,1735.000000
2,2022-01-01 03:00:00,2031.333333,1813.333333
3,2022-01-01 04:00:00,2915.666667,2327.666667
4,2022-01-01 05:00:00,3231.166667,2356.666667


## 8. 집계 결과 저장

다음 단계(일반화, 정규화, 평활화) 노트북에서 이어서 사용할 수 있도록 `agg/` 폴더에 CSV로 저장합니다.

In [9]:
ldaps_train_agg.to_csv(AGG_DIR / "ldaps_train_agg.csv", index=False, encoding="utf-8-sig")
gfs_train_agg.to_csv(AGG_DIR / "gfs_train_agg.csv", index=False, encoding="utf-8-sig")
ldaps_test_agg.to_csv(AGG_DIR / "ldaps_test_agg.csv", index=False, encoding="utf-8-sig")
gfs_test_agg.to_csv(AGG_DIR / "gfs_test_agg.csv", index=False, encoding="utf-8-sig")
scada_vestas_agg.to_csv(AGG_DIR / "scada_vestas_agg.csv", index=False, encoding="utf-8-sig")
scada_unison_agg.to_csv(AGG_DIR / "scada_unison_agg.csv", index=False, encoding="utf-8-sig")

for f in sorted(AGG_DIR.glob("*.csv")):
    print(f.name, f.stat().st_size, "bytes")

gfs_test_agg.csv 16176895 bytes
gfs_train_agg.csv 48586159 bytes
ldaps_test_agg.csv 11294700 bytes
ldaps_train_agg.csv 33981057 bytes
scada_unison_agg.csv 4558096 bytes
scada_vestas_agg.csv 16533101 bytes
